# Module 14: Advanced Matplotlib

## OO Interface, Annotations, Colormaps, Animation, and Performance

This module dives deep into the advanced features of Matplotlib. You will learn how to fully control every element of your figures using the **object-oriented (OO) interface**, customize styles globally, add annotations and arrows, create custom colormaps, embed images, animate data, and optimize performance — all using real **CHIRPS rainfall data** for Ethiopia.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings; warnings.filterwarnings('ignore')

ds = xr.open_dataset('../chirps_1981_2022.nc', engine='netcdf4')
precip = ds['precip']

# Ethiopia average
eth_ts = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0)).mean(dim=['lat', 'lon'])
rainfall_series = eth_ts.to_series()
rainfall_series.name = 'precip'
print(f'Loaded: {len(rainfall_series)} months, {rainfall_series.isna().sum()} NaN')

## 2. Object-Oriented Interface Deep Dive

The OO interface exposes the `Figure` and `Axes` objects, giving you granular control over every artist.

In [ ]:
# Full OO: construct a figure and axes explicitly
fig = plt.figure(figsize=(10, 5), dpi=100)

# Add a main axes using GridSpec for precise placement
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 1],
                      left=0.08, right=0.95, bottom=0.10, top=0.92,
                      wspace=0.25, hspace=0.30)
ax_main = fig.add_subplot(gs[0, 0])
ax_hist = fig.add_subplot(gs[0, 1])
ax_zoom = fig.add_subplot(gs[1, 0])
ax_stats = fig.add_subplot(gs[1, 1])

# Main time series
ax_main.plot(rainfall_series.index, rainfall_series.values, color='steelblue', lw=0.7)
ax_main.set_title('Ethiopia Monthly Rainfall (1981-2022)', loc='left')
ax_main.set_ylabel('mm/month')
ax_main.grid(True, alpha=0.3)

# Histogram
ax_hist.hist(rainfall_series.values, bins=40, orientation='horizontal',
             color='steelblue', edgecolor='white', alpha=0.7)
ax_hist.set_title('Distribution')
ax_hist.set_xlabel('Count')

# Zoom: 1995-2005
zoom = rainfall_series.loc['1995':'2005']
ax_zoom.fill_between(zoom.index, zoom.values, alpha=0.3, color='steelblue')
ax_zoom.plot(zoom.index, zoom.values, color='steelblue', lw=1)
ax_zoom.set_title('Zoom: 1995-2005')
ax_zoom.set_ylabel('mm/month')
ax_zoom.grid(True, alpha=0.3)

# Stats text box
stats_text = (
    f'Mean:  {rainfall_series.mean():.1f} mm\n'
    f'Std:   {rainfall_series.std():.1f} mm\n'
    f'Min:   {rainfall_series.min():.1f} mm\n'
    f'Max:   {rainfall_series.max():.1f} mm\n'
    f'Years: {rainfall_series.index.year.min()}-{rainfall_series.index.year.max()}'
)
ax_stats.text(0.1, 0.5, stats_text, transform=ax_stats.transAxes,
              fontsize=10, verticalalignment='center',
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax_stats.axis('off')
ax_stats.set_title('Statistics')

plt.show()

## 3. Custom rcParams and Style Sheets

Global and local style configuration for consistent, professional figures.

In [ ]:
# Save default rcParams, then apply a custom style context
print('Default figure size:', mpl.rcParams['figure.figsize'])
print('Available styles:', plt.style.available[:10])

# Use a style context manager
with plt.style.context('seaborn-v0_8-darkgrid'):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(rainfall_series.index, rainfall_series.values, lw=0.7)
    ax.set_title('Seaborn-darkgrid Style')
    ax.set_ylabel('mm/month')
plt.show()

In [ ]:
# Custom rcParams for journal style
custom_params = {
    'figure.dpi': 150,
    'figure.figsize': (7, 3.5),
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.linewidth': 0.8,
    'grid.alpha': 0.3,
    'axes.grid': True,
}

with plt.rc_context(custom_params):
    fig, ax = plt.subplots()
    ax.plot(rainfall_series.index, rainfall_series.values, color='black')
    ax.set_title('Custom Journal-Style rcParams')
    ax.set_xlabel('Date')
    ax.set_ylabel('Precipitation (mm/month)')
plt.show()

## 4. Annotations and Text Boxes

Use `ax.annotate()` and `ax.text()` to highlight events on your plots.

In [ ]:
# Annotate the 1984 drought and 1997 wet event
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rainfall_series.index, rainfall_series.values, color='gray', lw=0.6)

# Find extremes
driest = rainfall_series.idxmin()
wettest = rainfall_series.idxmax()
driest_val = rainfall_series.min()
wettest_val = rainfall_series.max()

# Annotate driest point
ax.annotate(f'Driest: {driest.date()} ({driest_val:.0f} mm)',
            xy=(driest, driest_val),
            xytext=(driest, driest_val + 40),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=9, color='red',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Annotate wettest point
ax.annotate(f'Wettest: {wettest.date()} ({wettest_val:.0f} mm)',
            xy=(wettest, wettest_val),
            xytext=(wettest, wettest_val - 50),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
            fontsize=9, color='blue',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Add informational text box
ax.text(0.02, 0.98, 'CHIRPS v2.0 | 0.05 deg', transform=ax.transAxes,
        fontsize=8, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))

ax.set_title('Annotated Extremes — Ethiopian Rainfall', fontsize=13)
ax.set_ylabel('mm/month')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Arrows and Shapes

Draw custom arrows, rectangles, and fancy boxes on your axes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rainfall_series.index, rainfall_series.values, color='steelblue', lw=0.7)

# FancyBboxPatch around the 1999-2003 period
bbox = FancyBboxPatch(
    (mdates.date2num(pd.Timestamp('1999-01-01')), 0),
    width=mdates.date2num(pd.Timestamp('2004-01-01')) - mdates.date2num(pd.Timestamp('1999-01-01')),
    height=200, boxstyle="round,pad=0.1",
    facecolor='yellow', alpha=0.2, edgecolor='orange', lw=2
)
ax.add_patch(bbox)
ax.text(mdates.date2num(pd.Timestamp('2000-06-01')), 180, 'Drought period (example)',
        fontsize=9, ha='center', color='darkorange', style='italic')

# Arrow pointing to a notable peak
peak_time = pd.Timestamp('2018-07-01')
peak_val = rainfall_series.loc[peak_time]
ax.annotate('', xy=(peak_time, peak_val), xytext=(peak_time, peak_val + 50),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.set_title('Arrows and Fancy Shapes on Rainfall Plot', fontsize=13)
ax.set_ylabel('mm/month')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Adding Images to Figures

In [ ]:
# Draw a rainfall map with imshow
jan_2020 = precip.sel(time='2020-01-01')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(jan_2020.values, extent=[30, 60, 0, 20], origin='upper',
               cmap='Blues', aspect='auto', interpolation='bilinear')
ax.set_title('CHIRPS Rainfall — Jan 2020 (imshow)', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
cbar = fig.colorbar(im, ax=ax, label='mm/month', shrink=0.85)
ax.grid(True, alpha=0.2, color='gray', linewidth=0.3)
plt.tight_layout()
plt.show()

## 7. Colormaps: Creation and Customization

In [ ]:
# Create a custom colormap for rainfall
colors = ['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84']
custom_cmap = LinearSegmentedColormap.from_list('rainfall_custom', colors, N=256)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(jan_2020.values, extent=[30, 60, 0, 20], origin='upper',
               cmap=custom_cmap, aspect='auto')
ax.set_title('Custom Rainfall Colormap', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
cbar = fig.colorbar(im, ax=ax, label='mm/month', shrink=0.85)
plt.tight_layout()
plt.show()

In [ ]:
# Reversed and truncated colormaps
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Original
gradient = np.linspace(0, 1, 256).reshape(1, -1)
axes[0].imshow(gradient, aspect='auto', cmap='Blues')
axes[0].set_title('Blues (original)')

# Reversed
axes[1].imshow(gradient, aspect='auto', cmap='Blues_r')
axes[1].set_title('Blues_r (reversed)')

# Truncated — use only middle 50%
from matplotlib.colors import truncate_colormap
trunc_blues = truncate_colormap(plt.cm.Blues, 0.25, 0.75)
axes[2].imshow(gradient, aspect='auto', cmap=trunc_blues)
axes[2].set_title('Blues (25%-75% truncated)')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 8. Colorbar Tweaking

Precise control over colorbar position, ticks, labels, and size.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(jan_2020.values, extent=[30, 60, 0, 20], origin='upper',
               cmap='viridis', aspect='auto')
ax.set_title('Colorbar Customization', fontsize=13)

# Custom colorbar with specific ticks and label
cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02,
                    ticks=[0, 50, 100, 150, 200, 250])
cbar.set_label('Precipitation (mm/month)', fontsize=11, labelpad=10)
cbar.ax.tick_params(labelsize=9)
cbar.ax.yaxis.set_ticks_position('left')

plt.tight_layout()
plt.show()

## 9. Animation Basics (FuncAnimation)

Animate the rainfall time series or a looping map sequence.

In [ ]:
# Animate: rainfall time series with a moving marker
fig, ax = plt.subplots(figsize=(10, 4))
x = rainfall_series.index.values
y = rainfall_series.values

line, = ax.plot([], [], color='steelblue', lw=0.8, label='Rainfall')
marker, = ax.plot([], [], 'ro', markersize=6)
ax.set_xlim(x[0], x[-1])
ax.set_ylim(y.min() - 20, y.max() + 20)
ax.set_title('Animated Rainfall Time Series')
ax.set_ylabel('mm/month')
ax.grid(True, alpha=0.3)
ax.legend()

def animate(frame):
    n = frame + 1
    line.set_data(x[:n], y[:n])
    marker.set_data([x[n-1]], [y[n-1]])
    return line, marker

ani = FuncAnimation(fig, animate, frames=min(len(x), 300), interval=20, blit=True, repeat=False)
plt.close(fig)  # prevent duplicate display
HTML(ani.to_jshtml())

In [ ]:
# Animate: rainfall maps over time (subset: 2000-2005)
time_subset = precip.sel(time=slice('2000-01-01', '2005-12-01'))
n_times = time_subset.shape[0]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(time_subset.isel(time=0).values, extent=[30, 60, 0, 20],
               origin='upper', cmap='Blues', aspect='auto', vmin=0, vmax=250)
cbar = fig.colorbar(im, ax=ax, label='mm/month', shrink=0.85)
ax.set_title('CHIRPS Rainfall Animation')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

def animate_maps(frame):
    im.set_array(time_subset.isel(time=frame).values)
    ax.set_title(f'CHIRPS Rainfall — {str(time_subset.time[frame].values)[:7]}')
    return im,

ani = FuncAnimation(fig, animate_maps, frames=n_times, interval=200, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

## 10. Performance Tips for Large Datasets

When plotting large CHIRPS arrays (400x600x504 = ~120M data points), use these strategies.

In [ ]:
# Tip 1: Downsample before plotting
decimated = rainfall_series[::12]  # every 12th point (annual)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

ax1.plot(rainfall_series.index, rainfall_series.values, lw=0.4)
ax1.set_title(f'Full series: {len(rainfall_series)} points', fontsize=10)

ax2.plot(decimated.index, decimated.values, 'o-', lw=0.8, markersize=3)
ax2.set_title(f'Decimated (annual): {len(decimated)} points', fontsize=10)

for ax in [ax1, ax2]:
    ax.set_ylabel('mm/month')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tip 2: Use rasterized lines for vector export
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rainfall_series.index, rainfall_series.values, lw=0.5, rasterized=True)
ax.set_title('Rasterized Line (smaller PDF/SVG files)', fontsize=11)
ax.set_ylabel('mm/month')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tip 3: Use pcolormesh with shading='nearest' for speed
# vs imshow or contourf
import time

data = precip.sel(time='2010-01-01').values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# pcolormesh
t0 = time.time()
axes[0].pcolormesh(ds.lon.values, ds.lat.values, data, shading='auto', cmap='Blues')
axes[0].set_title(f'pcolormesh: {time.time()-t0:.2f}s')

# contourf
t0 = time.time()
axes[1].contourf(ds.lon.values, ds.lat.values, data, levels=20, cmap='Blues')
axes[1].set_title(f'contourf: {time.time()-t0:.2f}s')

plt.tight_layout()
plt.show()

In [ ]:
# Tip 4: Aggregation before plotting
annual_mean = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0)).groupby('time.year').mean(dim='time')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(annual_mean.sel(year=2020).values, extent=[33, 48, 3, 15],
               origin='upper', cmap='YlGnBu', aspect='auto')
ax.set_title('Annual Mean Precipitation 2020 (aggregated)', fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
cbar = fig.colorbar(im, ax=ax, label='mm/year')
plt.tight_layout()
plt.show()

## 11. Exercises

1. **Drought Annotation**: Find the top 3 driest years in the CHIRPS record. Create a figure with the full time series and annotate each drought year with a text box and arrow explaining the year's rainfall total and the historical context.

2. **Custom Colormap**: Create a "rainfall intensity" colormap that goes from light yellow (dry) through green to dark blue (wet). Apply it to a map of Ethiopia's long-term mean annual rainfall.

3. **Animation**: Animate the monthly rainfall maps for the entire year 2015 (12 frames), showing the seasonal progression of rainfall over Ethiopia.

4. **Performance**: Compare the rendering speed of `imshow`, `pcolormesh`, and `contourf` for a full-resolution CHIRPS map. Which is fastest? Report results.

### Mini-Project

**Drought Event Dashboard**: Create a comprehensive figure that:
- Shows the full 1981-2022 rainfall time series 
- Highlights the 1984-1985 drought with a semi-transparent shaded rectangle
- Annotates the 3 driest months with arrows and text
- Adds a custom colorbar showing rainfall anomaly categories (severe dry, dry, normal, wet, severe wet)
- Uses a custom rcParams style for journal-ready appearance
- Exports the figure as both PNG (300 dpi) and PDF